# Semantic Search and RAG
- After the publication of this paper: https://arxiv.org/abs/1810.04805, Google announced it was using it to power Google search: https://blog.google/products-and-platforms/products/search/search-language-understanding-bert/

- Bing also stated that: Starting from April of this year, we used large transformer models to deliver the largest quality improvements to our Bing customers in the past year. Link: https://azure.microsoft.com/en-us/blog/bing-delivers-its-largest-improvement-in-search-experience-using-azure-gpus/




## Categories of Semantic search

- Dense Retrieval: it relies on the concept of the embeddings, and turn the search problem into retrieving the nearest neighbors of the search query (after both the query and documents are convererted to embeddings). Check the img1

- Reranking:  A reranking language model is one of these steps and is tasked with scoring the relevance of a subset of results against the query; the order of results is then changed based on these scores. Check img2

- RAG: Generative search is a subset of a broader type of category of systems better called RAG systems. These are text generation systems that incorporate search capabilities to reduce hallucinations, increase factuality, and/or ground the generation model on a specific dataset. Check img3

## Dense Retrieval
- Points that are close together mean that the text they represent is similar. 
- So in the img4, text 1 and text 2 are more similar to each other (because they are near each other) than text 3 (because it’s farther away). Check img4

- Should text 3 even be returned as a result? That’s a decision for you, the system designer. It’s sometimes desirable to have a max threshold of similarity score to filter out irrelevant results (in case the corpus has no relevant results for the query). check img5

- Are a query and its best result semantically similar? Not always. This is why language models need to be trained on question-answer pairs to become better at retrieval. chekc img5


### Dense Retrieval example

- We'll use Cohere to search the wikipedia page for the movie Interstellar:
  - Step1: Get the text we want to make searchable and apply some light processing to chunk it into sentences
  - Ste2: Embed each sentence
  - Step3: Build the search index
  - Step4: Search and see the results

In [1]:
pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-openai 1.1.13 requires langchain-core<2.0.0,>=1.2.29, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.



  Using cached langchain-0.2.5-py3-none-any.whl.metadata (7.0 kB)
  Using cached faiss_cpu-1.8.0-cp310-cp310-win_amd64.whl.metadata (3.8 kB)
  Using cached cohere-5.5.8-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_community-0.2.5-py3-none-any.whl.metadata (2.5 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl.metadata (10 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached fastavro-1.12.1-cp310-cp310-win_amd64.whl.metadata (5.7 kB)
  Using cached parameterized-0.9.0-py2.py3-none-any.whl.metadata (18 kB)
  Using cached types_requests-2.33.0.20260408-py3-none-any.whl.metadata (2.0 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata 

In [2]:
import cohere
api_key = "YEQiiMe07mAo5A3BMjJA7cDMYJ1Szbuv9eAKvi8t"
co = cohere.Client(api_key)



In [5]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [26]:
# Embedding the Text Chunks
import numpy as np

# Get the embeddings
response = co.embed(
  texts=texts,
  input_type="search_document",
  model="embed-v4.0"
).embeddings

embeds = np.array(response)
print(embeds.shape)

(15, 1536)


Before we can search, we need to build a search index.
-  An index stores the embeddings and is optimized to quickly retrieve the nearest neighbors even if we have a very large number of points:

In [27]:
# Build the search index
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))

We can now search the dataset using any query we want. We simply embed the query and present its embedding to the index, which will retrieve the most similar sentence from the Wikipedia article:

In [32]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

def search(query, number_of_results=3):

  # 1. Get the query's embedding
  query_embed = co.embed(texts=[query], 
                model="embed-v4.0",
                input_type="search_query",).embeddings[0]

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results

In [33]:
query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics,1.293341
1,"Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar",1.564170
2,"Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time",1.630263


- the first result has the least distance, and so the most similar to the query.
- Notice that this wouldn't have been possible if we were only doing keywords search because the top result did not include the same keywords in the query.
